# Sesión 2 — Qué gráfico corresponde a cada pregunta

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 2 de 9  
**Tipo:** Notebook guiado (caso del docente: Golden State Realty Advisors)

La S1 dejó la ficha del comité y una pregunta: **¿dónde la vivienda se vende por debajo de lo que el ingreso de sus
residentes sostiene?** Hoy se define el indicador que la responde y se recorre el directorio de visualizaciones:
para cada tipo de dato, el gráfico de referencia y sus alternativas. Este notebook reproduce cada cifra y cada
gráfico de la presentación, incluidas las galerías del directorio.

## Agenda de la sesión

| Parte | Contenido |
|---|---|
| Quiz | Quiz de la sesión 1 |
| Repaso | La ficha del comité y la pista de la S1 |
| Parte 1 | El indicador: cuántos años de ingreso cuesta la vivienda |
| Parte 2 | El directorio: cantidades, distribuciones, proporciones y múltiplos pequeños |
| Parte 3 | Relaciones x–y y puntos superpuestos |
| Parte 4 | Tiempo (y coordenadas polares) y geografía |
| Parte 5 | Para presentar a una audiencia y lo que conviene evitar |
| Parte 6 | Incertidumbre |
| Parte 7 | La primera lista de «gangas» |
| Parte 8 | El caso de los equipos: lo que dejó la S1 con CONECTA |

## Objetivos de aprendizaje

1. Definir un indicador que responda la pregunta del comité y distinguirlo de un KPI.
2. Elegir el gráfico según lo que se quiere mostrar: cantidades, distribuciones, proporciones, relaciones, tiempo, geografía o incertidumbre, y conocer las alternativas de cada tipo (Wilke, 2019, pp. 37–44; Knaflic, 2015, pp. 35–67).
3. Manejar la superposición de puntos con transparencia y histogramas en dos dimensiones (Wilke, 2019, pp. 219–225).
4. Mostrar la incertidumbre de una proporción con su intervalo de confianza, y reconocer sus condiciones y cuándo no se puede calcular (Diez et al., 2019, pp. 192–193 y 202–208; Wilke, 2019, pp. 186–196).
5. Comparar varios grupos con múltiplos pequeños: el mismo gráfico repetido por grupo, con la misma escala en todos los paneles (Wilke, 2019, pp. 255 y 258–259).

## Problemática de negocio

El comité de inversión de Golden State decide cada trimestre en qué distritos hace el equipo su debida diligencia
(capacidad: 150 distritos, un supuesto del caso que obliga a priorizar). Su pregunta, fijada en la S1: **¿dónde la
vivienda se vende por debajo de lo que el ingreso de sus residentes sostiene?** En palabras simples: ¿dónde la
vivienda es barata para lo que ganan quienes viven ahí? El equipo necesita un indicador que la responda y una
primera lista de candidatos. En esta sesión los datos todavía no se revisan: eso es la S3.

## Preparación del entorno

In [ ]:
import pathlib

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

def intervalo_proporcion(exitos, n, z=1.96):
    """Proporción con su intervalo del 95 % (OpenIntro, §5.2, p. 204) y la condición de éxito-fracaso (p. 205)."""
    p = exitos / n
    se = np.sqrt(p * (1 - p) / n)
    cumple = (exitos >= 10) & ((n - exitos) >= 10)
    return pd.DataFrame({"p": p, "inferior": p - z * se, "superior": p + z * se, "n": n, "cumple": cumple})

from scipy import stats
from matplotlib.patches import Rectangle


def rotular(ax, nombre, para):
    """Rótulo de una miniatura de la galería: nombre del gráfico y para qué sirve."""
    ax.set_title(f"{nombre}\n{para}", loc="left", fontsize=10.5, color=PALETA["texto"], linespacing=1.3)
    ax.tick_params(labelsize=8.5)

In [ ]:
FUENTE_GS = "Fuente: California Housing, censo de 1990 (Pace y Barry, 1997). Cálculos del curso."
FUENTE_CONECTA = "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026)."
df = pd.read_csv(DATOS / "housing.csv")
ZONAS = {"INLAND": "Interior", "<1H OCEAN": "A menos de 1 h del mar", "NEAR OCEAN": "Cerca del mar",
         "NEAR BAY": "Cerca de la bahía", "ISLAND": "Isla"}
CORTO = {"Interior": "Interior", "A menos de 1 h del mar": "< 1 h del mar", "Cerca del mar": "Cerca del mar",
         "Cerca de la bahía": "Bahía", "Isla": "Isla"}
df["zona"] = df["ocean_proximity"].map(ZONAS)
print(f"{len(df):,} distritos (datos crudos: la S3 los revisa)")

## Repaso de la sesión anterior

| Elemento de la ficha (S1) | Golden State |
|---|---|
| Destinatario | Comité de inversión |
| Decisión | Dónde hacer la debida diligencia del trimestre |
| Restricción | 150 distritos por trimestre: lo que alcanza a visitar el equipo (supuesto del caso); obliga a priorizar |
| Pregunta | ¿Dónde la vivienda se vende por debajo de lo que el ingreso de sus residentes sostiene? |

**La pista de la S1.** Al corregir el gráfico «malo» del informe (dos ejes), se vio que el valor de la vivienda
cambia mucho más entre zonas que el ingreso. Si el valor dependiera solo de lo que ganan los residentes, los dos
cambiarían en la misma proporción y una vivienda costaría los mismos años de ingreso en todas partes. No pasa: por eso
hay que comparar el valor con el ingreso, y la forma de compararlos es dividir.

In [ ]:
z4 = df[df["zona"] != "Isla"]
pista = z4.groupby("zona").agg(valor=("median_house_value", "mean"), ingreso=("median_income", "mean"))
pista["ingreso"] *= 10_000
pista["anios"] = pista["valor"] / pista["ingreso"]
veces = (pista[["valor", "ingreso"]] / pista.loc["Interior", ["valor", "ingreso"]]).sort_values("valor")   # interior = 1
fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(veces))
ax.barh(y + 0.2, veces["valor"], height=0.38, color=PALETA["principal"], label="Valor medio de la vivienda")
ax.barh(y - 0.2, veces["ingreso"], height=0.38, color=PALETA["neutro"], label="Ingreso medio del hogar")
for yi, (z, fila) in zip(y, veces.iterrows()):
    ax.text(fila["valor"] + 0.03, yi + 0.2, f"{decimal(fila['valor'], 1)} veces", va="center", fontsize=10.5,
            color=PALETA["principal"], fontweight="bold")
    ax.text(fila["ingreso"] + 0.03, yi - 0.2, f"{decimal(fila['ingreso'], 1)} veces", va="center", fontsize=10.5)
ax.axvline(1, color=PALETA["secundario"], linestyle="--", linewidth=1)
ax.set_yticks(y, [CORTO[z] for z in veces.index]); limpiar_ejes(ax); ax.set_xlim(0, veces.max().max() * 1.35)
ax.legend(frameon=False, fontsize=10.5, loc="lower right")
vb = veces.loc["Cerca de la bahía"]
titulo(fig, f"El valor cambia más que el ingreso: en la bahía, el valor es {decimal(vb['valor'], 1)} veces el del interior "
       f"y el ingreso, {decimal(vb['ingreso'], 1)}",
       "La pista de la S1: valor e ingreso medios por zona, como veces el del interior (línea: interior = 1; sin la Isla)")
fuente(fig, FUENTE_GS)
guardar(fig, "pista_s1.png")

## Parte 1 — El indicador: cuántos años de ingreso cuesta la vivienda

**Razón precio/ingreso** = valor mediano de la vivienda ÷ ingreso mediano anual del hogar (definición del curso).
Se lee en **años**: cuántos años de ingreso completo de un hogar típico del distrito cuesta la vivienda típica del
distrito. Una razón baja indica vivienda barata **para lo que ganan quienes viven ahí**, que es la pregunta de la
ficha. Ojo: en el censo el ingreso viene en decenas de miles de dólares; se multiplica por 10 000.

**Indicador y KPI.** Un indicador es una medida definida con su fórmula. Un KPI (indicador clave de rendimiento) es
«una indicación visual que comunica el progreso realizado para lograr un objetivo cuantificable» (Microsoft, 2026,
«KPI»): necesita además una meta y un umbral, y se vigila en el tiempo. La razón precio/ingreso es un indicador; no
llega a KPI porque el censo es una foto sin meta ni tiempo. En CONECTA, la tasa de cancelación se vuelve KPI en la S6,
cuando el tablero le pone una meta y la sigue mes a mes.

In [ ]:
df["razon_precio_ingreso"] = df["median_house_value"] / (df["median_income"] * 10_000)
# Un distrito real por zona: el de razón más cercana a la mediana de su zona.
ejemplo = {}
for z in ("Cerca de la bahía", "Interior"):
    d = df[df["zona"] == z]
    i = (d["razon_precio_ingreso"] - d["razon_precio_ingreso"].median()).abs().idxmin()
    ejemplo[z] = {"valor": d.loc[i, "median_house_value"], "ingreso": 10_000 * d.loc[i, "median_income"],
                  "razon": d.loc[i, "razon_precio_ingreso"]}
ejemplo = pd.DataFrame(ejemplo).T
for z, f in ejemplo.iterrows():
    print(f"{z}: USD {miles(f['valor'])} ÷ USD {miles(f['ingreso'])} = {decimal(f['razon'], 2)} años de ingreso")

**La regla de la «ganga».** Un distrito es «ganga» (candidato) si su razón está en el **5 % más bajo** de California.
El 5 % es una decisión del curso: un corte de cribado que da una lista más grande que la capacidad de 150 distritos,
porque la verificación de la S3 y la S4 descarta muchos. La S4 muestra que con el 4 % o el 6 % los mejores
candidatos son los mismos.

In [ ]:
umbral = df["razon_precio_ingreso"].quantile(0.05)
df["ganga"] = df["razon_precio_ingreso"] <= umbral
gangas = df[df["ganga"]]
pct_interior = 100 * (gangas["zona"] == "Interior").mean()
print(f"Razón mediana: {df['razon_precio_ingreso'].median():.2f} años; umbral del 5 %: {umbral:.2f}; "
      f"«gangas»: {len(gangas):,}, el {pct_interior:.0f} % en el interior")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
bins = np.linspace(0, 12, 61)
ax.hist(df["razon_precio_ingreso"].clip(upper=12), bins=bins, color=PALETA["neutro"])
ax.hist(gangas["razon_precio_ingreso"], bins=bins, color=PALETA["principal"])
ax.axvline(umbral, color=PALETA["marino"], linestyle="--", linewidth=1.1)
ax.text(umbral + 0.15, ax.get_ylim()[1] * 0.88, f"5 % más bajo: razón ≤ {decimal(umbral, 2)}\n{miles(len(gangas))} «gangas»",
        fontsize=10.5)
ax.set_xlabel("Razón precio/ingreso (años de ingreso que cuesta la vivienda; los mayores de 12 se agrupan en 12)")
ax.set_ylabel("Distritos")
titulo(fig, f"La vivienda típica de California cuesta {decimal(df['razon_precio_ingreso'].median(), 1)} años de ingreso",
       "Distribución de la razón precio/ingreso por distrito; en verde, el 5 % más bajo (datos sin revisar)")
fuente(fig, FUENTE_GS)
guardar(fig, "indicador.png")

## Parte 2 — El directorio: cantidades, distribuciones, proporciones y múltiplos pequeños

El directorio de Wilke (2019, cap. 5, pp. 37–44) se organiza por **lo que se quiere mostrar**. Para cada tipo hay un
gráfico de referencia y varias alternativas. En cada tipo, primero el gráfico de referencia en detalle; después, la
galería con las alternativas, con los datos del caso.

**Cantidades** (Wilke, 2019, pp. 37–38 y 45–58): barras desde cero y ordenadas (Knaflic, 2015, pp. 50–52); puntos;
barras agrupadas o apiladas cuando hay dos conjuntos de categorías; mapa de calor.

In [ ]:
mediana_zona = df.groupby("zona")["razon_precio_ingreso"].median().sort_values()
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(list(mediana_zona.index), mediana_zona.values, height=0.6,
        color=[PALETA["principal"] if z == "Interior" else PALETA["neutro"] for z in mediana_zona.index])
for z, v in mediana_zona.items():
    ax.text(v + 0.15, z, decimal(v, 2), va="center", fontsize=11)
limpiar_ejes(ax); ax.set_xlim(0, mediana_zona.max() * 1.15)
titulo(fig, "En el interior, la vivienda cuesta menos años de ingreso que en la costa",
       "Cantidades: razón precio/ingreso mediana por zona (barras desde cero); la Isla tiene 5 distritos")
fuente(fig, FUENTE_GS)
guardar(fig, "cantidades.png")

In [ ]:
# Galería de cantidades. Nivel de ingreso del distrito: terciles del ingreso mediano.
df["nivel_ingreso"], cortes = pd.qcut(df["median_income"], 3, labels=["Bajo", "Medio", "Alto"], retbins=True)
cortes = cortes * 10_000                       # en USD al año: los dos cortes separan los tres tercios de distritos
z4 = df[df["zona"] != "Isla"]
med = z4.groupby("zona")["razon_precio_ingreso"].median().sort_values()
tabla = z4.pivot_table(index="zona", columns="nivel_ingreso", values="razon_precio_ingreso", aggfunc="median",
                       observed=True).loc[med.index]
conteo = pd.crosstab(z4["zona"], z4["nivel_ingreso"]).loc[med.index]
nombres = [CORTO[z] for z in med.index]
tonos = [PALETA["neutro"], PALETA["principal_claro"], PALETA["principal"]]
fig, ejes = plt.subplots(2, 3, figsize=(12, 6.6))
fig.subplots_adjust(hspace=0.75, wspace=0.55)
a = ejes[0, 0]; a.barh(nombres, med.values, color=PALETA["principal"], height=0.6); a.set_xlim(0, 7)
a.set_xlabel("años de ingreso", fontsize=9.5); rotular(a, "Barras", "una cantidad por categoría; desde cero")
a = ejes[0, 1]; a.scatter(med.values, nombres, s=45, color=PALETA["principal"]); a.set_xlim(3, 6.5)
a.grid(axis="y", color=PALETA["fondo"]); a.set_xlabel("años de ingreso", fontsize=9.5)
rotular(a, "Puntos", "el eje no tiene que empezar en cero")
a = ejes[0, 2]; y = np.arange(len(med))
for k, (nivel, tono) in enumerate(zip(tabla.columns, tonos)):
    a.barh(y + (k - 1) * 0.27, tabla[nivel], height=0.27, color=tono, label=nivel)
a.set_yticks(y, nombres); a.set_xlabel("años de ingreso", fontsize=9.5)
a.legend(frameon=False, fontsize=8.5, ncol=3, loc="upper center", bbox_to_anchor=(0.45, -0.22),
         title="Ingreso del distrito", title_fontsize=8.5)
rotular(a, "Barras agrupadas", "dos conjuntos de categorías")
a = ejes[1, 0]; izq = np.zeros(len(conteo))
for nivel, tono in zip(conteo.columns, tonos):
    a.barh(nombres, conteo[nivel], left=izq, color=tono, height=0.6); izq += conteo[nivel].values
a.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: miles(v)))
a.set_xlabel("número de distritos", fontsize=9.5)
rotular(a, "Barras apiladas", "el total y sus partes (número de distritos)")
a = ejes[1, 1]; im = a.imshow(tabla.values, cmap="Greens", aspect="auto")
fig.colorbar(im, ax=a, shrink=0.85, label="años de ingreso")
a.set_xticks(range(3), list(tabla.columns)); a.set_yticks(range(len(med)), nombres)
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        a.text(j, i, decimal(tabla.values[i, j], 1), ha="center", va="center", fontsize=9.5,
               color="white" if tabla.values[i, j] > 5.5 else PALETA["texto"])
a.set_xlabel("ingreso del distrito", fontsize=9.5)
a.spines[:].set_visible(False); rotular(a, "Mapa de calor", "zona e ingreso; los años en el color")
a = ejes[1, 2]; a.bar(nombres, med.values, color=PALETA["principal"], width=0.6)
plt.setp(a.get_xticklabels(), rotation=25, ha="right")
a.set_ylabel("años de ingreso", fontsize=9.5); rotular(a, "Barras verticales", "pocas categorías con nombres cortos")
titulo(fig, "Cantidades: seis formas de mostrar la razón precio/ingreso mediana",
       f"Razón precio/ingreso mediana (años de ingreso) por zona, sin la Isla. Ingreso del distrito: bajo (menos de USD "
       f"{miles(cortes[1], -2)} al año), medio o alto (más de USD {miles(cortes[2], -2)}); las apiladas cuentan distritos",
       y=1.06)
fuente(fig, FUENTE_GS + " Wilke (2019), pp. 37–38; Knaflic (2015), pp. 36–37.")
guardar(fig, "galeria_cantidades.png")

**Distribuciones** (Wilke, 2019, pp. 38–39 y 59–92): histograma o densidad para una variable (los dos dependen de
parámetros y pueden engañar); acumulada y q-q, fieles pero más difíciles de leer; cajas, violines y franjas para
comparar muchos grupos; densidades superpuestas y ridgeline para comparar pocos o muchos.

In [ ]:
tope, piso = int((df["median_house_value"] >= 500_001).sum()), int((df["median_house_value"] <= 14_999).sum())
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(df["median_house_value"] / 1000, bins=60, color=PALETA["neutro"])
ax.annotate(f"{miles(tope)} distritos exactamente en USD 500 001:\n¿un valor real o un tope del censo?",
            xy=(495, tope * 0.95), xytext=(470, tope * 0.95), fontsize=10.5, va="center", ha="right",
            arrowprops={"arrowstyle": "->", "color": PALETA["acento"]})
ax.set_xlabel("Valor mediano de la vivienda (miles de USD)"); ax.set_ylabel("Distritos")
titulo(fig, "La distribución del valor tiene un pico extraño en el extremo",
       "Distribución: histograma del valor mediano por distrito (datos sin revisar)")
fuente(fig, FUENTE_GS)
guardar(fig, "distribucion_valor.png")
print(f"Valores en el tope: {tope}; en el piso (14 999): {piso}")

In [ ]:
orden = mediana_zona.drop("Isla").index
fig, ax = plt.subplots(figsize=(9, 4))
datos = [df.loc[df["zona"] == z, "razon_precio_ingreso"].clip(upper=15) for z in orden]
partes = ax.violinplot(datos, vert=False, showmedians=True, showextrema=False, widths=0.8)
for cuerpo, z in zip(partes["bodies"], orden):
    cuerpo.set_facecolor(PALETA["principal"] if z == "Interior" else PALETA["neutro"]); cuerpo.set_alpha(0.85)
partes["cmedians"].set_color(PALETA["marino"]); partes["cmedians"].set_linewidth(1.5)
ax.axvline(umbral, color=PALETA["acento"], linestyle="--", linewidth=1.2)
ax.text(umbral - 0.1, len(orden) + 0.35, "umbral del 5 %", ha="right", fontsize=10, color=PALETA["acento"])
ax.set_yticks(range(1, len(orden) + 1), list(orden)); ax.set_xlabel("Razón precio/ingreso (años de ingreso)")
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
titulo(fig, "Casi todo el interior está más cerca del umbral que la costa",
       "Distribuciones: violines de la razón precio/ingreso por zona (sin la Isla; valores mayores de 15 se agrupan)")
fuente(fig, FUENTE_GS)
guardar(fig, "violines.png")

In [ ]:
r = df["razon_precio_ingreso"].clip(upper=12)
muestra = r.sample(4000, random_state=1)
malla = np.linspace(0, 12, 300)
zonas4 = ["Interior", "A menos de 1 h del mar", "Cerca del mar", "Cerca de la bahía"]
por_zona = {z: df.loc[df["zona"] == z, "razon_precio_ingreso"].clip(upper=12) for z in zonas4}
densidad = {z: stats.gaussian_kde(v.sample(min(len(v), 3000), random_state=1))(malla) for z, v in por_zona.items()}
color_z = {z: (PALETA["principal"] if z == "Interior" else t) for z, t in
           zip(zonas4, [None, PALETA["neutro"], PALETA["principal_claro"], PALETA["acento"]])}
fig, ejes = plt.subplots(3, 3, figsize=(14.5, 8.6))
fig.subplots_adjust(hspace=0.8, wspace=0.35)
a = ejes[0, 0]; a.hist(r, bins=40, color=PALETA["principal"]); rotular(a, "Histograma", "una variable; depende del ancho")
a = ejes[0, 1]; a.fill_between(malla, stats.gaussian_kde(muestra)(malla), color=PALETA["principal"], alpha=0.8)
rotular(a, "Densidad", "una variable; depende del suavizado")
a = ejes[0, 2]; a.plot(np.sort(r), np.linspace(0, 1, len(r)), color=PALETA["principal"], linewidth=2)
rotular(a, "Distribución acumulada", "fiel a los datos; más difícil de leer")
a = ejes[1, 0]; q = np.sort(muestra); teo = stats.norm.ppf((np.arange(1, len(q) + 1) - 0.5) / len(q))
a.scatter(teo, q, s=3, color=PALETA["principal"]); a.plot(teo, q.mean() + q.std() * teo, color=PALETA["acento"], linewidth=1.2)
a.set_xlabel("valor esperado si fuera normal", fontsize=9.5); a.set_ylabel("razón observada (años)", fontsize=9.5)
a.text(-3.6, 12.6, "tope de 12 (valores agrupados)", fontsize=8.5, color=PALETA["secundario"])
rotular(a, "Gráfico q-q", "si los puntos se curvan, no es normal")
a = ejes[1, 1]; a.boxplot([por_zona[z] for z in zonas4], vert=False, widths=0.6, flierprops={"markersize": 1.5},
                          medianprops={"color": PALETA["acento"], "linewidth": 1.8})
a.set_yticks(range(1, 5), [CORTO[z] for z in zonas4]); rotular(a, "Cajas", "muchos grupos; resumen y atípicos")
a = ejes[1, 2]; v = a.violinplot([por_zona[z] for z in zonas4], vert=False, showmedians=True, showextrema=False)
for cuerpo in v["bodies"]:
    cuerpo.set_facecolor(PALETA["principal"]); cuerpo.set_alpha(0.7)
v["cmedians"].set_color(PALETA["marino"])
a.set_yticks(range(1, 5), [CORTO[z] for z in zonas4]); rotular(a, "Violines", "muchos grupos; la forma completa")
a = ejes[2, 0]; rng = np.random.default_rng(1)
for k, z in enumerate(zonas4):
    m = por_zona[z].sample(250, random_state=1)
    a.scatter(m, k + rng.uniform(-0.25, 0.25, len(m)), s=3, color=PALETA["principal"], alpha=0.5)
a.set_yticks(range(4), [CORTO[z] for z in zonas4]); rotular(a, "Franjas (strip)", "cada observación, con ruido vertical")
a = ejes[2, 1]
for z in zonas4:
    a.fill_between(malla, densidad[z], color=color_z[z], alpha=0.35); a.plot(malla, densidad[z], color=color_z[z], linewidth=1)
rotular(a, "Densidades superpuestas", "pocos grupos en el mismo eje")
a = ejes[2, 2]
for k, z in enumerate(zonas4[::-1]):
    a.fill_between(malla, k, k + densidad[z] / max(d.max() for d in densidad.values()) * 1.6, color=color_z[z], alpha=0.85)
    a.text(0, k + 0.35, CORTO[z], ha="left", fontsize=8.5)
a.set_yticks([]); rotular(a, "Ridgeline", "muchos grupos, desplazados en vertical")
for a in ejes.flat:
    a.spines["left"].set_visible(a not in (ejes[2, 2],))
titulo(fig, "Distribuciones: nueve formas de mostrar cómo se reparte la razón precio/ingreso",
       "Galería del directorio con Golden State; los valores mayores de 12 se agrupan en 12", y=1.03)
fuente(fig, FUENTE_GS + " Wilke (2019), pp. 38–39 y 78–79 (q-q).")
guardar(fig, "galeria_distribuciones.png")

**Proporciones** (Wilke, 2019, pp. 39–40): torta (resalta fracciones simples de un todo), barras lado a lado (se
comparan mejor), barras apiladas (varios conjuntos de proporciones), densidades apiladas (proporciones a lo largo de
una variable continua), mosaico y treemap (proporciones anidadas en dos variables) y conjuntos paralelos (tres o más variables; Wilke,
2019, pp. 113–116). El **diagrama de Sankey** es la versión de los conjuntos paralelos para **flujos**: el ancho de
cada banda es la cantidad que pasa de una categoría a otra.

In [ ]:
cz = z4["zona"].value_counts().loc[med.index[::-1]]
pz_niv = 100 * pd.crosstab(z4["nivel_ingreso"], z4["zona"], normalize="index")[med.index[::-1]]
tonos4 = [PALETA["principal"], PALETA["principal_claro"], PALETA["neutro"], PALETA["acento"]]
fig, ejes = plt.subplots(2, 3, figsize=(12, 6.6))
fig.subplots_adjust(hspace=0.7, wspace=0.45)
a = ejes[0, 0]; a.pie(cz.values, labels=[CORTO[z] for z in cz.index], colors=tonos4, startangle=90, counterclock=False,
                      textprops={"fontsize": 8.5}, wedgeprops={"edgecolor": "white"})
rotular(a, "Torta", "partes de un todo; fracciones simples")
a = ejes[0, 1]; a.bar([CORTO[z] for z in cz.index], 100 * cz / cz.sum(), color=tonos4, width=0.6)
a.tick_params(axis="x", rotation=25); a.set_ylabel("% de distritos", fontsize=9.5)
rotular(a, "Barras lado a lado", "las partes se comparan mejor")
a = ejes[0, 2]; izq = np.zeros(3)
for z, tono in zip(pz_niv.columns, tonos4):
    a.barh(list(pz_niv.index), pz_niv[z], left=izq, color=tono, height=0.6, label=CORTO[z]); izq += pz_niv[z].values
a.set_xlim(0, 100); a.set_ylabel("Nivel de ingreso", fontsize=9.5)
rotular(a, "Barras apiladas", "varios conjuntos de proporciones")
a = ejes[1, 0]; tramo = pd.qcut(z4["median_income"], 20)
pd_ing = pd.crosstab(tramo, z4["zona"], normalize="index")[med.index[::-1]]
centro = z4.groupby(tramo, observed=True)["median_income"].median() * 10
a.stackplot(centro.values, *[100 * pd_ing[z].values for z in pd_ing.columns], colors=tonos4)
a.set_xlabel("Ingreso mediano (miles de USD)", fontsize=9.5); a.set_ylim(0, 100)
rotular(a, "Densidades apiladas", "proporciones a lo largo de una variable")
a = ejes[1, 1]; x0 = 0
pn = z4["nivel_ingreso"].value_counts(normalize=True).loc[["Bajo", "Medio", "Alto"]]
for nivel, ancho in pn.items():
    y0 = 0
    for z, tono in zip(pz_niv.columns, tonos4):
        alto = pz_niv.loc[nivel, z] / 100
        a.add_patch(Rectangle((x0, y0), ancho, alto, facecolor=tono, edgecolor="white", linewidth=1.5)); y0 += alto
    a.text(x0 + ancho / 2, -0.07, nivel, ha="center", va="top", fontsize=9.5); x0 += ancho
a.set_xlim(0, 1); a.set_ylim(0, 1); a.set_axis_off(); rotular(a, "Mosaico", "dos variables: ancho y alto")
a = ejes[1, 2]; x0 = 0; tot = len(z4)
for z, tono in zip(cz.index, tonos4):
    ancho = cz[z] / tot; y0 = 0
    for nivel in ["Bajo", "Medio", "Alto"]:
        alto = conteo.loc[z, nivel] / cz[z]
        a.add_patch(Rectangle((x0, y0), ancho, alto, facecolor=tono, edgecolor="white", linewidth=1.5,
                              alpha={"Bajo": 1, "Medio": 0.75, "Alto": 0.5}[nivel])); y0 += alto
    if ancho > 0.25:
        a.text(x0 + 0.01, 0.97, CORTO[z], va="top", fontsize=8.5, color="white")
    x0 += ancho
a.set_xlim(0, 1); a.set_ylim(0, 1); a.set_axis_off(); rotular(a, "Treemap", "anidadas: zona (color) y nivel de ingreso (tono)")
fig.legend([Rectangle((0, 0), 1, 1, color=t) for t in tonos4], [CORTO[z] for z in cz.index], loc="lower center",
           ncol=4, frameon=False, fontsize=9.5, bbox_to_anchor=(0.5, -0.03))
titulo(fig, "Proporciones: seis formas de mostrar cómo se reparten los distritos por zona",
       "Galería del directorio con Golden State (sin la Isla); nivel de ingreso: terciles del ingreso mediano", y=1.06)
fig.text(0.01, -0.08, FUENTE_GS + " Wilke (2019), pp. 39–40.", fontsize=8.5, color=PALETA["secundario"])
guardar(fig, "galeria_proporciones.png")

In [ ]:
# Conjuntos paralelos con flujos (Sankey): de qué segmento es cada cliente del año y cómo termina (CONECTA).
hs = pd.read_csv(DATOS / "conecta_clientes_mes.csv", dtype={"mes": str}).sort_values("mes")
primera = hs.groupby("cliente_id").head(1).set_index("cliente_id")
ultima = hs.groupby("cliente_id").tail(1).set_index("cliente_id")
sigue = ultima["mes"] == hs["mes"].max()
destino = pd.Series(np.where(sigue, "Siguen", ultima["tipo_cancelacion"]), index=ultima.index)
valido = sigue | (ultima["cancelo"] == 1)          # un cliente sale sin registro de cancelación: queda fuera
seg_cliente = pd.Series(np.where(primera["linea"] == "Hogar", "Hogar",
                                 np.where(primera["modalidad"] == "Prepago", "Móvil prepago", "Móvil pospago")),
                        index=primera.index)
flujo = pd.crosstab(seg_cliente[valido], destino[valido])
IZQ, DER = ["Hogar", "Móvil pospago", "Móvil prepago"], ["Siguen", "Voluntaria", "Por mora", "Sin recarga"]
flujo = flujo.reindex(index=IZQ, columns=DER, fill_value=0)
COLOR = {"Siguen": PALETA["neutro"], "Voluntaria": PALETA["acento"], "Por mora": PALETA["marino"],
         "Sin recarga": PALETA["principal"]}
total, hueco = flujo.values.sum(), 0.03 * flujo.values.sum()


def nodos(tamanos):
    # Posición (arriba, abajo) de cada nodo, apilados de arriba hacia abajo con un hueco entre ellos.
    pos, cursor = {}, total + hueco * (len(tamanos) - 1)
    for nombre, n in tamanos.items():
        pos[nombre] = [cursor, cursor - n]; cursor -= n + hueco
    return pos


pi, pd_ = nodos(flujo.sum(axis=1)), nodos(flujo.sum(axis=0))
libre_i = {k: v[0] for k, v in pi.items()}; libre_d = {k: v[0] for k, v in pd_.items()}
fig, ax = plt.subplots(figsize=(12, 6))
xs = np.linspace(0, 1, 100); suave = 3 * xs ** 2 - 2 * xs ** 3
for s_ in IZQ:
    for d_ in DER:
        n = flujo.loc[s_, d_]
        if n == 0:
            continue
        a0, a1 = libre_i[s_], libre_d[d_]
        arriba = a0 + (a1 - a0) * suave; abajo = (a0 - n) + ((a1 - n) - (a0 - n)) * suave
        ax.fill_between(xs, abajo, arriba, color=COLOR[d_], alpha=0.45 if d_ == "Siguen" else 0.8, linewidth=0)
        libre_i[s_] -= n; libre_d[d_] -= n
for nombre, (arr, aba) in pi.items():
    ax.add_patch(Rectangle((-0.03, aba), 0.03, arr - aba, color=PALETA["texto"]))
    ax.text(-0.05, (arr + aba) / 2, f"{nombre}\n{miles(arr - aba)} clientes", ha="right", va="center", fontsize=10.5)
for nombre, (arr, aba) in pd_.items():
    ax.add_patch(Rectangle((1, aba), 0.03, arr - aba, color=COLOR[nombre]))
    ax.text(1.05, (arr + aba) / 2, f"{nombre}: {miles(arr - aba)} ({decimal(100 * (arr - aba) / total, 1)} %)",
            va="center", fontsize=10.5, color=PALETA["texto"])
ax.set_xlim(-0.45, 1.5); ax.set_axis_off()
pct_pre = 100 * flujo.loc["Móvil prepago"].drop("Siguen").sum() / flujo.loc["Móvil prepago"].sum()
pct_hog = 100 * flujo.loc["Hogar"].drop("Siguen").sum() / flujo.loc["Hogar"].sum()
titulo(fig, f"En el año se va el {decimal(pct_pre, 0)} % de los clientes móviles prepago y el {decimal(pct_hog, 0)} % de los de "
       "Hogar; la mayoría, por decisión propia",
       "Sankey (conjuntos paralelos con flujos): segmento de cada cliente del año y cómo termina; el ancho de cada banda "
       "es el número de clientes (CONECTA, datos sin revisar)", y=1.04)
fuente(fig, FUENTE_CONECTA + " Conjuntos paralelos: Wilke (2019), pp. 113–116.")
guardar(fig, "sankey.png")

**Múltiplos pequeños** (Wilke, 2019, pp. 255 y 258–259): el mismo gráfico repetido en una cuadrícula, un panel por
grupo. Todos los paneles usan la misma escala; si cada uno tuviera la suya, grupos muy distintos parecerían iguales.

In [ ]:
# Múltiplos pequeños: el mismo histograma por zona, con la misma escala en los cuatro paneles.
# Cada zona tiene un número distinto de distritos: el eje y es el porcentaje de los distritos de la zona.
bins = np.linspace(0, 12, 49)
fig, ejes = plt.subplots(2, 2, figsize=(11, 5.6), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.35, wspace=0.06)
for ax, z in zip(ejes.flat, zonas4):
    rz = df.loc[df["zona"] == z, "razon_precio_ingreso"].clip(upper=12)
    ax.hist(rz, bins=bins, weights=np.full(len(rz), 100 / len(rz)),
            color=PALETA["principal"] if z == "Interior" else PALETA["neutro"])
    ax.axvline(umbral, color=PALETA["acento"], linestyle="--", linewidth=1.1)
    ax.set_title(f"{z}: {miles(len(rz))} distritos, {decimal(100 * (rz <= umbral).mean(), 1)} % bajo el umbral",
                 loc="left", fontsize=11)
for ax in ejes[1]:
    ax.set_xlabel("Razón precio/ingreso (años; 12 o más se agrupan en 12)")
for ax in ejes[:, 0]:
    ax.set_ylabel("% de los distritos")
titulo(fig, "Con la misma escala en los cuatro paneles, casi todo lo que queda bajo el umbral está en el interior",
       "Múltiplos pequeños: distribución de la razón precio/ingreso por zona; en naranja, el umbral del 5 % (sin la Isla)",
       y=1.06)
fuente(fig, FUENTE_GS)
guardar(fig, "multiples.png")

### Punto de control 1 — Cantidades y distribuciones

**¿Por qué una barra sirve para la mediana por zona, pero no para mostrar cómo se reparten los distritos?**
La barra resume un solo número; la distribución necesita histograma, cajas o violines.

**¿Qué pregunta abre el pico en USD 500 001?**
Si es un valor real o un tope del censo. Se resuelve en la S3.

> ¿Cambiaría la conclusión si se usara el promedio en lugar de la mediana?

## Parte 3 — Relaciones x–y y puntos superpuestos

La **dispersión** es el gráfico de dos variables cuantitativas (Wilke, 2019, pp. 41 y 117–121); con una tercera
variable en el tamaño, burbujas; con más de dos variables, el correlograma. Con miles de puntos aparece la
**superposición**: los puntos se tapan. Se maneja con transparencia, contornos o un histograma en dos dimensiones
(Wilke, 2019, pp. 41 y 219–225).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
x, y = df["median_income"] * 10, df["median_house_value"] / 1000
a1.scatter(x, y, s=6, color=PALETA["marino"])
a1.set_title("Sin transparencia: los puntos se tapan", loc="left", fontsize=11.5)
hb = a2.hexbin(x, y, gridsize=45, cmap="Greens", mincnt=1, bins="log")
a2.set_title("Histograma en dos dimensiones: se ve la densidad", loc="left", fontsize=11.5)
for ax in (a1, a2):
    ax.set_xlabel("Ingreso mediano (miles de USD)")
a1.set_ylabel("Valor mediano (miles de USD)")
titulo(fig, f"Con {miles(len(df))} distritos, la dispersión necesita otro tratamiento",
       "Relaciones x–y: el mismo dato con puntos sólidos y con un histograma en dos dimensiones", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "superposicion.png")

In [ ]:
x_all, y_all = df["median_income"] * 10, df["median_house_value"] / 1000
s = df.sample(2500, random_state=1)
xs, ys = s["median_income"] * 10, s["median_house_value"] / 1000
fig, ejes = plt.subplots(2, 3, figsize=(12, 6.8))
fig.subplots_adjust(hspace=0.85, wspace=0.45)
a = ejes[0, 0]; a.scatter(xs, ys, s=3, color=PALETA["marino"]); rotular(a, "Dispersión", "dos variables cuantitativas")
a = ejes[0, 1]; a.scatter(x_all, y_all, s=3, color=PALETA["marino"], alpha=0.06)
rotular(a, "Dispersión con transparencia", "las zonas densas se ven oscuras")
b = s.sample(250, random_state=2)
a = ejes[0, 2]; a.scatter(b["median_income"] * 10, b["median_house_value"] / 1000, s=b["population"] / 25,
                          color=PALETA["principal"], alpha=0.5, edgecolor="white", linewidth=0.4)
rotular(a, "Burbujas", "una tercera variable en el tamaño (población)")
a = ejes[1, 0]; a.hexbin(x_all, y_all, gridsize=35, cmap="Greens", mincnt=1, bins="log")
rotular(a, "Hexágonos", "cuenta los puntos por celda")
a = ejes[1, 1]; kde = stats.gaussian_kde(np.vstack([xs, ys]))
gx, gy = np.meshgrid(np.linspace(0, 150, 80), np.linspace(0, 520, 80))
a.contour(gx, gy, kde(np.vstack([gx.ravel(), gy.ravel()])).reshape(gx.shape), levels=8, cmap="Greens", linewidths=1.2)
rotular(a, "Contornos de densidad", "dónde se concentran los puntos")
for a in ejes.flat[:5]:
    a.set_xlim(0, 150); a.set_ylim(0, 540)
    a.set_xlabel("ingreso del hogar (miles de USD al año)", fontsize=9.5); a.set_ylabel("valor de la vivienda (miles de USD)", fontsize=9.5)
ejes[0, 0].text(3, 515, "tope del censo: USD 500 001", fontsize=8.5, color=PALETA["acento"])
a = ejes[1, 2]
variables = {"Ingreso": "median_income", "Valor": "median_house_value", "Edad": "housing_median_age",
             "Población": "population", "Hogares": "households"}
corr = df[list(variables.values())].corr().values
a.imshow(corr, cmap="BrBG", vmin=-1, vmax=1)
a.set_xticks(range(5), list(variables), rotation=45, fontsize=8.5); a.set_yticks(range(5), list(variables), fontsize=8.5)
for i in range(5):
    for j in range(5):
        a.text(j, i, decimal(round(corr[i, j], 1) + 0.0, 1), ha="center", va="center", fontsize=8.5,
               color="white" if abs(corr[i, j]) > 0.6 else PALETA["texto"])
a.spines[:].set_visible(False); rotular(a, "Correlograma", "las correlaciones de varias variables")
titulo(fig, "Relaciones x–y: seis formas de mostrar el ingreso frente al valor de la vivienda",
       "Cada punto es un distrito: ingreso mediano del hogar (x) frente a valor mediano de la vivienda (y); las burbujas "
       "agregan la población; el correlograma compara cinco variables de a pares",
       y=1.05)
fuente(fig, FUENTE_GS + " Wilke (2019), p. 41.")
guardar(fig, "galeria_relaciones.png")

## Parte 4 — Tiempo y geografía

**Tiempo** (Wilke, 2019, pp. 42 y 131–160): el eje x es el tiempo y los puntos se unen con líneas; con dos variables
en el tiempo, la dispersión conectada; con mucho ruido, una línea suavizada. El censo es una foto de un solo momento:
no tiene tiempo, así que esta galería usa la serie mensual de CONECTA (el caso de los equipos, datos sin revisar).

**Coordenadas polares** (Wilke, 2019, pp. 22–24): el ángulo es el tiempo y la distancia al centro, el valor. Sirven
para datos **periódicos**, en los que el final del ciclo se une con el principio (la hora del día, el día de la
semana, los meses de varios años). Usadas en una serie que no se repite, unen el último dato con el primero como si
fueran vecinos.

In [ ]:
por_edad = df.groupby("housing_median_age")["median_house_value"].mean() / 1000
edad_52 = int((df["housing_median_age"] >= 52).sum())
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(por_edad.index, por_edad.values, color=PALETA["principal"], linewidth=2)
ax.annotate(f"{miles(edad_52)} distritos en 52 años:\n¿otro tope del censo?", xy=(52, por_edad.loc[52]),
            xytext=(30, por_edad.max() * 1.02), fontsize=10.5, arrowprops={"arrowstyle": "->", "color": PALETA["acento"]})
ax.set_xlabel("Edad mediana de la vivienda (años)"); ax.set_ylabel("Valor medio (miles de USD)")
ax.set_ylim(0, por_edad.max() * 1.2)
titulo(fig, "Esto no es una serie de tiempo: es la edad de las viviendas en un solo censo",
       "Valor medio por edad de la vivienda; un pico en 52 años")
fuente(fig, FUENTE_GS)
guardar(fig, "no_es_tiempo.png")

In [ ]:
h = pd.read_csv(DATOS / "conecta_clientes_mes.csv", dtype={"mes": str})
por_mes = h.groupby("mes").agg(cancelaciones=("cancelo", "sum"), filas=("cancelo", "size"), quejas=("quejas_mes", "mean"))
tasa_mes = 100 * por_mes["cancelaciones"] / por_mes["filas"]
MESES = [f"{m[5:]}/{m[2:4]}" for m in tasa_mes.index]
xm = np.arange(len(tasa_mes))
fig, ejes = plt.subplots(2, 2, figsize=(13.5, 7))
fig.subplots_adjust(hspace=0.65, wspace=0.3)
a = ejes[0, 0]; a.plot(xm, tasa_mes, color=PALETA["principal"], linewidth=2.2, marker="o", markersize=4)
a.set_xticks(xm[::3], MESES[::3]); a.set_ylim(0, 4); a.set_xlabel("mes", fontsize=9.5)
a.set_ylabel("tasa de cancelación (%)", fontsize=9.5); rotular(a, "Líneas", "una variable en el tiempo")
a = ejes[0, 1]; qx, ty = por_mes["quejas"].values, tasa_mes.values
a.scatter(qx, ty, s=18, color=PALETA["principal"], zorder=3)
for i in range(len(qx) - 1):                     # flechas: el sentido del tiempo
    a.annotate("", xy=(qx[i + 1], ty[i + 1]), xytext=(qx[i], ty[i]),
               arrowprops={"arrowstyle": "->", "color": PALETA["principal"], "linewidth": 1.1})
for i in (0, 6, len(xm) - 1):
    a.annotate(MESES[i], (qx[i], ty[i]), xytext=(6, -12), textcoords="offset points", fontsize=8.5)
a.set_xlabel("quejas promedio por cliente en el mes", fontsize=9.5); a.set_ylabel("tasa de cancelación (%)", fontsize=9.5)
rotular(a, "Dispersión conectada", "dos variables mes a mes (el eje no empieza en cero)")
a = ejes[1, 0]; a.scatter(xm, tasa_mes, s=18, color=PALETA["neutro"])
a.plot(xm, tasa_mes.rolling(3, center=True, min_periods=2).mean(), color=PALETA["principal"], linewidth=2.2)
a.set_xticks(xm[::3], MESES[::3]); a.set_ylim(0, 4); a.set_xlabel("mes", fontsize=9.5)
a.set_ylabel("tasa de cancelación (%)", fontsize=9.5)
rotular(a, "Línea suavizada", "la tendencia sin el ruido (media móvil de 3 meses)")
ejes[1, 1].remove()
a = fig.add_subplot(2, 2, 4, projection="polar")
ang = 2 * np.pi * xm / 12
a.plot(np.append(ang, ang[0]), np.append(tasa_mes.values, tasa_mes.values[0]), color=PALETA["acento"], linewidth=2)
a.set_xticks(ang, MESES, fontsize=8.5); a.set_yticks([1, 2, 3], ["1", "2", "3"], fontsize=8.5); a.set_ylim(0, 4)
a.set_theta_offset(np.pi / 2); a.set_theta_direction(-1)
a.set_title("Coordenadas polares\nsolo si el dato es cíclico", loc="left", fontsize=10.5, color=PALETA["texto"])
titulo(fig, "Tiempo: cuatro formas de mostrar la cancelación mensual de CONECTA",
       "El censo no tiene tiempo: la galería usa CONECTA, tasa de cancelación mensual de nov. 2025 a oct. 2026 "
       "(datos sin revisar)",
       y=1.04)
fuente(fig, FUENTE_CONECTA + " Wilke (2019), pp. 22–24 y 42.")
guardar(fig, "galeria_tiempo.png")

In [ ]:
# Cuándo no usar coordenadas polares: doce meses de un solo año no son un ciclo que se repite.
fig = plt.figure(figsize=(11, 4.2))
a1 = fig.add_subplot(1, 2, 1)
a1.plot(xm, tasa_mes, color=PALETA["principal"], linewidth=2.2, marker="o", markersize=4)
a1.set_xticks(xm[::2], MESES[::2]); a1.set_ylim(0, 4); a1.set_ylabel("Cancelación mensual registrada (%)")
a1.set_title("Cartesiano: el tiempo avanza de izquierda a derecha", loc="left", fontsize=11.5)
a2 = fig.add_subplot(1, 2, 2, projection="polar")
a2.plot(ang, tasa_mes.values, color=PALETA["principal"], linewidth=2.2, marker="o", markersize=4)
a2.plot([ang[-1], 2 * np.pi], [tasa_mes.values[-1], tasa_mes.values[0]], color=PALETA["acento"], linewidth=2.2, linestyle="--")
a2.set_xticks(ang, MESES, fontsize=9.5); a2.set_yticks([1, 2, 3], ["1", "2", "3"], fontsize=9.5); a2.set_ylim(0, 4)
a2.set_theta_offset(np.pi / 2); a2.set_theta_direction(-1)
a2.set_title("Polar: el círculo une oct. 26 con nov. 25 (en naranja)", loc="left", fontsize=11.5)
titulo(fig, "Doce meses de un solo año no son un ciclo: en polares, octubre de 2026 parece vecino de noviembre de 2025",
       "La misma serie en coordenadas cartesianas y polares; las polares sirven para lo que se repite (hora, día de la semana, meses de varios años)",
       y=1.1)
fuente(fig, FUENTE_CONECTA + " Wilke (2019), pp. 22–24.")
guardar(fig, "polares.png")

**Geografía** (Wilke, 2019, pp. 42 y 161–180): puntos sobre un mapa proyectado (S1) o regiones coloreadas según un
dato (coropleta). Los cartogramas, que deforman las regiones según otra cantidad, no se muestran. Los mapas son el
centro de la S5.

**Las regiones del curso.** Para ubicar las «gangas» se agrupan los condados de California en tres regiones (Bahía de
San Francisco, Sur de California y Valle Central) y el resto del estado. Es una definición del curso, la misma que
usan la S4 y la S5. El **Valle Central** son 17 condados del centro del estado, entre la costa y la Sierra Nevada:
de Kern, Fresno y Tulare (al sur) a Sacramento, Butte y Shasta (al norte).

In [ ]:
REGIONES = {
    "Bahía de San Francisco": ["Alameda", "Contra Costa", "Santa Clara", "San Francisco", "San Mateo",
                               "Marin", "Sonoma", "Napa", "Solano"],
    "Sur de California": ["Los Angeles", "Orange", "San Diego", "Imperial", "Riverside", "San Bernardino", "Ventura"],
    "Valle Central": ["Kern", "Fresno", "Tulare", "Kings", "Madera", "Merced", "Stanislaus", "San Joaquin",
                      "Sacramento", "Yolo", "Sutter", "Yuba", "Butte", "Glenn", "Colusa", "Tehama", "Shasta"]}
region_de = {co: r for r, cs in REGIONES.items() for co in cs}
condados = gpd.read_file(DATOS / "cb_2018_us_county_20m.zip")
california = condados[condados["STATEFP"] == "06"].to_crs(3310)[["NAME", "geometry"]].rename(columns={"NAME": "condado"})
california["region"] = california["condado"].map(region_de).fillna("Resto del estado")
puntos = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326").to_crs(3310)
puntos = gpd.sjoin_nearest(puntos, california, how="left")
puntos = puntos[~puntos.index.duplicated()]          # un distrito, un condado (el más cercano)
por_region = pd.DataFrame({"distritos_pct": 100 * puntos["region"].value_counts(normalize=True),
                           "gangas_pct": 100 * puntos.loc[puntos["ganga"], "region"].value_counts(normalize=True),
                           "tasa_pct": 100 * puntos.groupby("region")["ganga"].mean()}).sort_values("gangas_pct")
display(por_region.round(1))

In [ ]:
# El mapa para el comité: dónde está el Valle Central y cuántas «gangas» tiene.
valle = california[california["region"] == "Valle Central"].dissolve()
bahia = california[california["region"] == "Bahía de San Francisco"].dissolve()
la = california[california["condado"] == "Los Angeles"]
fig = plt.figure(figsize=(12, 6.2))
ax = fig.add_axes([0.0, 0.0, 0.5, 0.92])
california.plot(ax=ax, color=PALETA["fondo"], edgecolor="white", linewidth=0.6)
ax.scatter(puntos.loc[~puntos["ganga"]].geometry.x, puntos.loc[~puntos["ganga"]].geometry.y, s=1, color=PALETA["neutro"], alpha=0.35)
ax.scatter(puntos.loc[puntos["ganga"]].geometry.x, puntos.loc[puntos["ganga"]].geometry.y, s=4, color=PALETA["principal"])
valle.boundary.plot(ax=ax, color=PALETA["acento"], linewidth=2)
for geo, nombre, dx, dy, color in ((valle, "Valle Central", 160_000, 60_000, PALETA["acento"]),
                                   (bahia, "Bahía de\nSan Francisco", -230_000, -40_000, PALETA["secundario"]),
                                   (la, "Los Ángeles", -60_000, -170_000, PALETA["secundario"])):
    p = geo.geometry.iloc[0].representative_point()
    ax.annotate(nombre, (p.x, p.y), xytext=(p.x + dx, p.y + dy), fontsize=11, color=color, fontweight="bold",
                ha="center", arrowprops={"arrowstyle": "-", "color": color, "linewidth": 0.8})
ax.set_axis_off()
ax.set_title("En verde, las «gangas»; en naranja, el contorno del Valle Central", loc="left", fontsize=11.5)
ab = fig.add_axes([0.62, 0.12, 0.36, 0.62])
yr = np.arange(len(por_region))
ab.barh(yr + 0.18, por_region["distritos_pct"], height=0.34, color=PALETA["neutro"], label="% de los distritos")
ab.barh(yr - 0.18, por_region["gangas_pct"], height=0.34, color=PALETA["principal"], label="% de las «gangas»")
for yi, (_, f) in zip(yr, por_region.iterrows()):
    ab.text(f["distritos_pct"] + 1, yi + 0.18, f"{f['distritos_pct']:.0f} %", va="center", fontsize=10)
    ab.text(f["gangas_pct"] + 1, yi - 0.18, f"{f['gangas_pct']:.0f} %", va="center", fontsize=10, color=PALETA["principal"],
            fontweight="bold")
ab.set_yticks(yr, list(por_region.index)); limpiar_ejes(ab); ab.set_xlim(0, 85)
ab.legend(frameon=False, fontsize=10, loc="lower right")
ab.set_title("Cada región: su parte de los distritos y de las «gangas»", loc="left", fontsize=11.5)
vc = por_region.loc["Valle Central"]
titulo(fig, f"El {vc['gangas_pct']:.0f} % de las «gangas» está en el Valle Central, que tiene el {vc['distritos_pct']:.0f} % "
       "de los distritos", "Geografía: puntos sobre un mapa proyectado de California (proyección de áreas iguales, "
       "EPSG:3310); regiones del curso", y=1.04)
fuente(fig, FUENTE_GS + " Límites: U.S. Census Bureau (2019).")
guardar(fig, "mapa_gangas.png")

In [ ]:
# Galería de geografía: puntos y coropleta (porcentaje de «gangas» por condado).
pc = puntos.groupby("condado")["ganga"].mean().mul(100).rename("pct")
mapa_c = california.merge(pc, left_on="condado", right_index=True, how="left")
fig, ejes = plt.subplots(1, 2, figsize=(11, 5.6))
fig.subplots_adjust(wspace=0.05)
a = ejes[0]; california.plot(ax=a, color=PALETA["fondo"], edgecolor="white")
a.scatter(puntos.loc[~puntos["ganga"]].geometry.x, puntos.loc[~puntos["ganga"]].geometry.y, s=1, color=PALETA["neutro"], alpha=0.5)
a.scatter(puntos.loc[puntos["ganga"]].geometry.x, puntos.loc[puntos["ganga"]].geometry.y, s=3, color=PALETA["principal"])
rotular(a, "Mapa de puntos", "cada distrito en su lugar; en verde, las «gangas»")
a = ejes[1]; mapa_c.plot(ax=a, column="pct", cmap="Greens", edgecolor="white", linewidth=0.4, legend=True,
                         legend_kwds={"shrink": 0.6, "label": "% de distritos que son «ganga»"})
rotular(a, "Coropleta", "cada región por un dato normalizado (%);\nlos condados con pocos distritos exageran (S5)")
for a in ejes:
    a.set_axis_off()
titulo(fig, "Geografía: dos formas de mostrar dónde están las «gangas»",
       "Galería del directorio con Golden State; los cartogramas (regiones deformadas por otra cantidad) no se muestran",
       y=1.1)
fuente(fig, FUENTE_GS + " Límites: U.S. Census Bureau (2019). Wilke (2019), p. 42.")
guardar(fig, "galeria_geografia.png")

## Parte 5 — Para presentar a una audiencia y lo que conviene evitar

Knaflic (2015, cap. 2) propone pocos gráficos para comunicar a una audiencia de negocio: el **texto simple** cuando
hay uno o dos números (p. 38), la **tabla** cuando la audiencia busca su fila (p. 40), la **pendiente** para comparar
dos momentos (p. 47), la **cascada** para mostrar cómo se llega de un total a otro (p. 55), las **barras
horizontales** como gráfico de referencia (p. 57) y el **área cuadrada** para proporciones pequeñas (p. 60). Y pide
evitar la torta, la dona, el 3D y el eje secundario (pp. 61–67).

In [ ]:
# Para la pendiente y la cascada se usa CONECTA: el censo no tiene dos momentos.
grandes = h[h["localidad"].isin(set(h["localidad"]) & set(gpd.read_file(DATOS / "upz.geojson")["localidad"]))]
seis = grandes.groupby("localidad")["cliente_id"].nunique().nlargest(6).index
inicio = 100 * grandes[grandes["mes"] <= "2026-01"].groupby("localidad")["cancelo"].mean().loc[seis]
final = 100 * grandes[grandes["mes"] >= "2026-08"].groupby("localidad")["cancelo"].mean().loc[seis]
m0, m1 = h["mes"].min(), h["mes"].max()
activos_ini, activos_fin = h.loc[h["mes"] == m0, "cliente_id"].nunique(), h.loc[h["mes"] == m1, "cliente_id"].nunique()
entradas = int((h.groupby("cliente_id")["mes"].min() > m0).sum())
salidas = int((h.groupby("cliente_id")["mes"].max() < m1).sum())
assert activos_ini + entradas - salidas == activos_fin
fig, ejes = plt.subplots(2, 3, figsize=(12, 7))
fig.subplots_adjust(hspace=0.6, wspace=0.4)
a = ejes[0, 0]; a.set_axis_off()
a.text(0.02, 0.55, decimal(df["razon_precio_ingreso"].median(), 1), fontsize=44, fontweight="bold", color=PALETA["principal"])
a.text(0.02, 0.25, "años de ingreso cuesta la vivienda\ntípica de California", fontsize=11, color=PALETA["texto"])
rotular(a, "Texto simple (Golden State)", "uno o dos números: no hace falta un gráfico")
a = ejes[0, 1]; a.set_axis_off()
tab = pd.DataFrame({"Razón mediana": mediana_zona.drop("Isla").map(lambda v: decimal(v, 2)),
                    "«Gangas»": df[df["zona"] != "Isla"].groupby("zona")["ganga"].mean().map(lambda v: f"{decimal(100 * v, 1)} %")})
t = a.table(cellText=tab.values, rowLabels=[CORTO[z] for z in tab.index], colLabels=list(tab.columns), loc="center",
            cellLoc="center")
t.auto_set_font_size(False); t.set_fontsize(9.5); t.scale(1, 1.5)
for (i, j), celda in t.get_celld().items():
    celda.set_edgecolor(PALETA["fondo"])
rotular(a, "Tabla (Golden State)", "la audiencia busca su fila")
a = ejes[0, 2]
y_rot, previo = {}, -1e9
for loc in final.sort_values().index:                 # rótulos separados al menos 0,06 puntos
    y_rot[loc] = max(final[loc], previo + 0.06); previo = y_rot[loc]
for loc in seis:
    destacada = final[loc] - inicio[loc] == (final - inicio).max()
    color = PALETA["acento"] if destacada else PALETA["neutro"]
    a.plot([0, 1], [inicio[loc], final[loc]], color=color, marker="o", linewidth=2 if destacada else 1.2)
    a.text(1.05, y_rot[loc], loc.title(), va="center", fontsize=8.5, color=color)
a.set_xticks([0, 1], ["nov.–ene.", "ago.–oct."]); a.set_xlim(-0.15, 1.7); a.set_ylabel("tasa de cancelación (%)", fontsize=9.5)
rotular(a, "Pendiente (CONECTA)", "dos momentos; quién sube y quién baja")
a = ejes[1, 0]
pasos = [("Activos\nnov. 25", activos_ini, 0), ("Entran", entradas, activos_ini), ("Salen", -salidas, activos_ini + entradas),
         ("Activos\noct. 26", activos_fin, 0)]
for k, (nombre, v, base) in enumerate(pasos):
    color = PALETA["neutro"] if k in (0, 3) else (PALETA["principal"] if v > 0 else PALETA["acento"])
    a.bar(k, v, bottom=base, color=color, width=0.6)
    a.text(k, base + max(v, 0) + 400, miles(abs(v)), ha="center", fontsize=9.5)
a.set_xticks(range(4), [p[0] for p in pasos], fontsize=8.5); a.set_ylim(0, (activos_ini + entradas) * 1.12)
a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: miles(v))); a.set_ylabel("clientes", fontsize=9.5)
rotular(a, "Cascada (CONECTA)", "cómo se llega de un total a otro")
a = ejes[1, 1]
pct_int = round(100 * df.loc[df["zona"] == "Interior", "ganga"].mean())
for k in range(100):
    fila, col = divmod(k, 10)
    a.add_patch(Rectangle((col, 9 - fila), 0.88, 0.88, color=PALETA["principal"] if k < pct_int else PALETA["fondo"]))
a.set_xlim(0, 10); a.set_ylim(0, 10); a.set_aspect("equal"); a.set_axis_off()
a.text(5, -0.6, f"De cada 100 distritos del interior, {pct_int} son «gangas»;\nen la costa, menos de 2", ha="center",
       va="top", fontsize=9.5)
rotular(a, "Área cuadrada (Golden State)", "una proporción, de un vistazo")
a = ejes[1, 2]; pg = 100 * df[df["zona"] != "Isla"].groupby("zona")["ganga"].mean().loc[med.index]
a.barh([CORTO[z] for z in pg.index], pg, color=PALETA["principal"], height=0.6)
a.barh([CORTO[z] for z in pg.index], 100 - pg, left=pg, color=PALETA["fondo"], height=0.6)
for z, v in pg.items():
    a.text(v + 1.5, CORTO[z], f"{decimal(v, 1)} %", va="center", fontsize=8.5)
a.set_xlim(0, 100); a.set_xlabel("% de los distritos de la zona", fontsize=9.5)
rotular(a, "Barras horizontales apiladas (Golden State)", "la parte frente al todo, por categoría")
titulo(fig, "Para presentar a una audiencia: seis gráficos de Knaflic con los datos del curso",
       "Golden State y CONECTA (pendiente y cascada: el censo no tiene dos momentos); datos sin revisar", y=1.05)
fuente(fig, FUENTE_GS + " " + FUENTE_CONECTA.replace("Fuente: ", "") + " Knaflic (2015), pp. 38–60.")
guardar(fig, "galeria_audiencia.png")

In [ ]:
fig = plt.figure(figsize=(13.5, 7))
fig.subplots_adjust(hspace=0.75, wspace=0.45)
a = fig.add_subplot(2, 2, 1)
top = gangas.merge(puntos[["condado"]], left_index=True, right_index=True)["condado"].value_counts()
partes = pd.concat([top.iloc[:11], pd.Series({"Otros": top.iloc[11:].sum()})])
a.pie(partes.values, labels=partes.index, textprops={"fontsize": 8.5}, startangle=90, wedgeprops={"edgecolor": "white"})
rotular(a, "Evitar: torta con muchas partes (Golden State)",
        "«gangas» por condado: no se comparan 12 ángulos (p. 61)\nMejor: barras ordenadas")
a = fig.add_subplot(2, 2, 2)
a.pie(cz.values, labels=[CORTO[z] for z in cz.index], colors=tonos4, startangle=90, textprops={"fontsize": 8.5},
      wedgeprops={"width": 0.35, "edgecolor": "white"})
rotular(a, "Evitar: dona (Golden State)", "distritos por zona: se comparan arcos (p. 65)\nMejor: barras lado a lado")
a = fig.add_subplot(2, 2, 3, projection="3d")
a.bar3d(np.arange(len(med)), np.zeros(len(med)), np.zeros(len(med)), 0.6, 0.6, med.values, color=PALETA["principal_claro"])
a.set_xticks(np.arange(len(med)) + 0.3, nombres, fontsize=8.5); a.set_yticks([]); a.tick_params(labelsize=8.5)
a.set_title("Evitar: 3D (Golden State)\nrazón mediana por zona: la perspectiva deforma (p. 65)\nMejor: barras de cantidades",
            loc="left", fontsize=10.5, color=PALETA["texto"])
a = fig.add_subplot(2, 2, 4)
act = h.groupby("mes")["cliente_id"].nunique()
a.bar(xm, act.values, color=PALETA["neutro"]); a.set_ylim(28_000, 30_200)
b = a.twinx(); b.plot(xm, tasa_mes.values, color=PALETA["acento"], linewidth=2, marker="o", markersize=4); b.set_ylim(0, 3.4)
a.set_xticks(xm[::3], MESES[::3]); a.tick_params(labelsize=8.5); b.tick_params(labelsize=8.5)
a.set_ylabel("clientes activos (eje desde 28 000)", fontsize=9.5); b.set_ylabel("tasa de cancelación (%)", fontsize=9.5)
rotular(a, "Evitar: eje secundario (CONECTA)", "las escalas deciden la relación (p. 67)\nMejor: un panel por variable (S1)")
titulo(fig, "Lo que conviene evitar: cuatro gráficos que dificultan la lectura",
       "Knaflic (2015, pp. 61–67) con los datos del curso; la torta y el eje secundario son los del informe de la S1", y=1.04)
fuente(fig, FUENTE_GS + " " + FUENTE_CONECTA.replace("Fuente: ", ""))
guardar(fig, "galeria_evitar.png")

**Para buscar más gráficos.** El directorio de hoy no los agota. El **Data Viz Project** (Ferdio, s. f.) es un
catálogo de más de 150 visualizaciones que se filtra por función (comparación, distribución, parte de un todo,
tendencia, correlación, geolocalización), por forma y por tipo de dato: sirve para encontrar el nombre de un gráfico y
ver ejemplos. Es un catálogo de inspiración de una agencia, no una fuente académica. Regla del curso: encuéntrenlo
ahí; decidan cuándo usarlo con Wilke y Knaflic, y citen la fuente. https://datavizproject.com/

## Parte 6 — Incertidumbre

Un **intervalo de confianza** acompaña una cifra estimada con una **muestra**: dice entre qué valores está, con 95 %
de confianza, la cifra de toda la población. Es **estimación ± 1,96 × SE**, con $SE = \sqrt{\hat{p}(1-\hat{p})/n}$
(Diez et al., 2019, §5.2, p. 204), y tiene dos condiciones (Diez et al., 2019, pp. 192–193 y 205):

1. **Independencia:** las observaciones vienen de una muestra aleatoria.
2. **Éxito-fracaso:** al menos 10 éxitos y 10 fracasos.

**«95 % de confianza»** quiere decir que, si se repite el muestreo muchas veces, cerca del 95 % de los intervalos
contienen el valor real (Diez et al., 2019, pp. 202–203). Como tenemos todos los distritos, podemos comprobarlo: 25
muestras aleatorias de 500 distritos del interior, y cada intervalo frente a la cifra real.

**Con todos los datos** no hay error de muestreo: la proporción de «gangas» de cada zona es la cifra exacta de 1990.
Lo que sí importa es el **tamaño de cada grupo**: con pocos casos, una cifra es inestable y no sirve para concluir.
Las barras de error muestran el intervalo en el gráfico (Wilke, 2019, pp. 43 y 186–196).

In [ ]:
# Con todos los distritos: la proporción exacta de «gangas» por zona y el tamaño de cada zona (n).
por_zona_g = df.groupby("zona")["ganga"].agg(["sum", "size"])
prop = intervalo_proporcion(por_zona_g["sum"], por_zona_g["size"])
display(prop[["p", "n"]].assign(p=(100 * prop["p"]).round(2)).rename(columns={"p": "% de «gangas»", "n": "distritos"}))

In [ ]:
# Qué significa «95 % de confianza»: 25 muestras aleatorias de 500 distritos del interior (semilla fija).
SEMILLA = 2026
interior = df[df["zona"] == "Interior"].reset_index(drop=True)
rng = np.random.default_rng(SEMILLA)
exitos = [int(interior.loc[rng.choice(len(interior), 500, replace=False), "ganga"].sum()) for _ in range(25)]
muestras = intervalo_proporcion(pd.Series(exitos), 500)
real_interior = interior["ganga"].mean()
muestras["contiene"] = (muestras["inferior"] <= real_interior) & (real_interior <= muestras["superior"])
fig, ax = plt.subplots(figsize=(9, 5.2))
for k, fila in muestras.iterrows():
    color = PALETA["principal"] if fila["contiene"] else PALETA["acento"]
    ax.hlines(k, 100 * fila["inferior"], 100 * fila["superior"], color=color, linewidth=2 if fila["contiene"] else 3)
    ax.scatter(100 * fila["p"], k, color=color, s=20, zorder=3)
ax.axvline(100 * real_interior, color=PALETA["marino"], linestyle="--", linewidth=1.2)
ax.text(100 * real_interior + 0.2, len(muestras) - 0.2, f"valor real: {decimal(100 * real_interior, 1)} %", fontsize=10.5,
        color=PALETA["marino"])
ax.set_yticks([]); ax.set_ylabel("25 muestras aleatorias de 500 distritos")
ax.set_xlabel("% de «gangas» en el interior, con intervalo del 95 %")
ax.spines["left"].set_visible(False)
titulo(fig, f"{muestras['contiene'].sum()} de los 25 intervalos contienen el valor real: eso es «95 % de confianza»",
       "Incertidumbre: 25 muestras aleatorias de 500 distritos del interior; en naranja, los que no lo contienen")
fuente(fig, FUENTE_GS + " Como en Diez et al. (2019), pp. 202–203 (Figura 5.6).")
guardar(fig, "intervalos_muestras.png")

In [ ]:
# Con todos los datos no hay error de muestreo: lo que pesa es el tamaño de cada zona.
tam = prop.assign(pct=100 * prop["p"]).sort_values("n")
colores = [PALETA["acento"] if n < 30 else PALETA["principal"] for n in tam["n"]]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
fig.subplots_adjust(wspace=0.08)
a1.barh(list(tam.index), tam["pct"], color=colores, height=0.6)
for z, fila in tam.iterrows():
    a1.text(fila["pct"] + 0.3, z, f"{decimal(fila['pct'], 1)} %", va="center", fontsize=10.5)
limpiar_ejes(a1); a1.set_xlim(0, tam["pct"].max() * 1.3)
a1.set_title("% de distritos que son «gangas» (cifra exacta)", loc="left", fontsize=11.5)
a2.barh(list(tam.index), tam["n"], color=colores, height=0.6)
for z, fila in tam.iterrows():
    a2.text(fila["n"] + 120, z, miles(fila["n"]), va="center", fontsize=10.5)
limpiar_ejes(a2); a2.set_yticks([]); a2.set_xlim(0, tam["n"].max() * 1.25)
a2.set_title("Distritos de la zona", loc="left", fontsize=11.5)
titulo(fig, "Con todos los distritos no hay error de muestreo: lo que pesa es el tamaño de cada zona",
       f"La Isla tiene {int(tam.loc['Isla', 'n'])} distritos: su 0 % no permite concluir nada; las demás zonas tienen miles",
       y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "incertidumbre.png")

In [ ]:
# Galería de incertidumbre (Wilke, 2019, pp. 43–44): el mismo dato con muestras de 100, 250 y 600 distritos del interior,
# dibujado de cinco formas, y la banda de confianza de una serie (CONECTA, 3 000 filas por mes).
rng2 = np.random.default_rng(SEMILLA + 2)   # muestras de ejemplo que contienen el valor real (la figura de 25 muestras muestra una que no)
tamanos = [100, 250, 600]
ex = [int(interior.loc[rng2.choice(len(interior), n, replace=False), "ganga"].sum()) for n in tamanos]
ri = intervalo_proporcion(pd.Series(ex, index=tamanos), pd.Series(tamanos, index=tamanos))
se = np.sqrt(ri["p"] * (1 - ri["p"]) / ri["n"])
yi, nom = np.arange(len(ri)), [f"n = {n}" for n in tamanos]
fig, ejes = plt.subplots(2, 3, figsize=(13.5, 7))
fig.subplots_adjust(hspace=0.75, wspace=0.4)
a = ejes[0, 0]; a.hlines(yi, 100 * ri["inferior"], 100 * ri["superior"], color=PALETA["principal"], linewidth=2.5)
a.scatter(100 * ri["p"], yi, color=PALETA["marino"], zorder=3, s=25); rotular(a, "Barras de error", "el intervalo alrededor del punto")
a = ejes[0, 1]
for z_, grosor in ((2.576, 1.5), (1.96, 3.5), (1.282, 6)):
    a.hlines(yi, 100 * (ri["p"] - z_ * se), 100 * (ri["p"] + z_ * se), color=PALETA["principal"], linewidth=grosor)
a.scatter(100 * ri["p"], yi, color="white", edgecolor=PALETA["marino"], zorder=3, s=22)
rotular(a, "Barras de error graduadas", "80 %, 95 % y 99 % a la vez")
a = ejes[0, 2]
for z_ in np.linspace(0.1, 2.6, 18):
    a.hlines(yi, 100 * (ri["p"] - z_ * se), 100 * (ri["p"] + z_ * se), color=PALETA["principal"], linewidth=10, alpha=0.09)
rotular(a, "Franjas de confianza", "la incertidumbre en el degradado")
a = ejes[1, 0]
for k, (n, fila) in enumerate(ri.iterrows()):
    xg = np.linspace(fila["p"] - 3.2 * se[n], fila["p"] + 3.2 * se[n], 120)
    dens = stats.norm.pdf(xg, fila["p"], se[n]); dens = 0.4 * dens / dens.max()
    a.fill_between(100 * xg, k - dens, k + dens, color=PALETA["principal_claro"], alpha=0.8)
    a.hlines(k, 100 * fila["inferior"], 100 * fila["superior"], color=PALETA["marino"], linewidth=2)
rotular(a, "Ojos", "los valores plausibles y el intervalo")
a = ejes[1, 1]; a.barh(yi, 100 * ri["p"], color=PALETA["neutro"], height=0.6)
a.errorbar(100 * ri["p"], yi, xerr=[100 * (ri["p"] - ri["inferior"]), 100 * (ri["superior"] - ri["p"])], fmt="none",
           ecolor=PALETA["marino"], elinewidth=1.5, capsize=3)
rotular(a, "Barras con error", "la cantidad y su intervalo")
for a in list(ejes[0]) + [ejes[1, 0], ejes[1, 1]]:
    a.axvline(100 * real_interior, color=PALETA["acento"], linestyle="--", linewidth=1)
    a.set_yticks(yi, nom); a.set_xlim(0, 30); a.set_ylabel("tamaño de la muestra", fontsize=9.5)
    a.set_xlabel("% de «gangas» en el interior", fontsize=9.5)
mm = h.groupby("mes", group_keys=False).sample(3000, random_state=SEMILLA)
cm = mm.groupby("mes")["cancelo"].agg(["sum", "size"])
im = intervalo_proporcion(cm["sum"], cm["size"])
a = ejes[1, 2]
a.fill_between(xm, 100 * im["inferior"], 100 * im["superior"], color=PALETA["principal_claro"], alpha=0.4)
a.plot(xm, 100 * im["p"], color=PALETA["principal"], linewidth=2, label="con la muestra")
a.plot(xm, tasa_mes.values, color=PALETA["acento"], linewidth=1.2, linestyle="--", label="real")
a.set_xticks(xm[::3], MESES[::3]); a.set_ylim(1.5, 4.5); a.legend(frameon=False, fontsize=8.5, loc="upper left")
a.set_xlabel("mes", fontsize=9.5); a.set_ylabel("tasa de cancelación (%)", fontsize=9.5)
rotular(a, "Banda de confianza (CONECTA)", "el intervalo a lo largo de una línea")
titulo(fig, "Incertidumbre: seis formas de mostrar un intervalo; con más casos, el intervalo se estrecha",
       f"% de «gangas» del interior con muestras aleatorias de 100, 250 y 600 distritos (línea naranja: valor real, "
       f"{decimal(100 * real_interior, 1)} %); banda: CONECTA con 3 000 filas por mes", y=1.06)
fuente(fig, FUENTE_GS + " " + FUENTE_CONECTA.replace("Fuente: ", "") + " Wilke (2019), pp. 43–44.")
guardar(fig, "galeria_incertidumbre.png")

### Punto de control 2 — Relaciones, geografía e incertidumbre

**¿Por qué la Isla, con 5 distritos, no sirve para concluir?**
Con tan pocos casos, su cifra (0 %) es inestable: un solo distrito la cambiaría en 20 puntos. Además, si fuera una
muestra, no cumpliría la condición de éxito-fracaso.

**¿Qué muestra el mapa que no muestran las barras por zona?**
Que las «gangas» se concentran en una región, el Valle Central: tiene una parte pequeña de los distritos y la mayoría
de las «gangas». La zona y la región podrían explicarlas (S4).

> ¿Qué pasaría con los intervalos si las muestras fueran de 50 distritos?

## Parte 7 — La primera lista de «gangas»

Con el indicador y los gráficos, el equipo tiene una primera lista. Antes de llevarla al comité hay que revisarla.

In [ ]:
display(gangas.nsmallest(5, "razon_precio_ingreso")[["zona", "population", "median_income", "median_house_value",
                                                      "razon_precio_ingreso"]])
print(f"Las dos mejores «gangas» del estado valen exactamente USD {miles(gangas.nsmallest(2, 'razon_precio_ingreso')['median_house_value'].iloc[0])}")

**La tensión que pasa a la S3.** La lista tiene 1 032 «gangas», el 92 % en el interior. Además:

- las dos mejores valen exactamente USD 14 999;
- 965 distritos están en USD 500 001;
- 1 273 tienen exactamente 52 años.

¿Son datos reales o límites del censo? La S3 responde si se puede confiar en lo que vemos.

## Parte 8 — El caso de los equipos: lo que dejó la S1 con CONECTA

Los tres hallazgos del taller de la S1, en una figura: las diferencias entre localidades son pequeñas (el eje
recortado del informe las exageraba), la cancelación es estable y los datos todavía no están limpios (el informe dejó
fuera las filas con el nombre de la localidad mal escrito). Son el punto de partida del taller de hoy.

In [ ]:
upz_loc = gpd.read_file(DATOS / "upz.geojson").set_index("codigo_upz")["localidad"]
OFICIALES = set(upz_loc)
activos = h.groupby("mes")["cliente_id"].nunique()
cancelacion = 100 * h.groupby("mes")["cancelo"].mean()
of = h[h["localidad"].isin(OFICIALES)]
por_localidad = 100 * of.groupby("localidad")["cancelo"].mean().sort_values()
promedio_ciudad = 100 * of["cancelo"].mean()
real = por_localidad["KENNEDY"] / por_localidad["CHAPINERO"]
aparente = (por_localidad["KENNEDY"] - 2.5) / (por_localidad["CHAPINERO"] - 2.5)
fuera = ~h["localidad"].isin(OFICIALES)
fuera_loc = h.loc[fuera, "codigo_upz"].map(upz_loc).value_counts()
formas = h.assign(oficial=h["codigo_upz"].map(upz_loc)).groupby("oficial")["localidad"].nunique()
fig = plt.figure(figsize=(15, 6.8))
gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1, 1], hspace=0.6, wspace=0.55)
a1 = fig.add_subplot(gs[:, 0])
colores = [PALETA["acento"] if l in ("KENNEDY", "CHAPINERO") else PALETA["neutro"] for l in por_localidad.index]
a1.barh([l.title() for l in por_localidad.index], por_localidad.values, color=colores, height=0.65)
a1.axvline(promedio_ciudad, color=PALETA["marino"], linestyle="--", linewidth=1)
a1.text(promedio_ciudad + 0.05, -1.3, f"promedio: {decimal(promedio_ciudad, 2)} %", fontsize=9.5, color=PALETA["marino"])
a1.text(0.1, len(por_localidad) + 0.3, f"Kennedy cancela {decimal(real, 1)} veces lo que Chapinero; con el eje\n"
        f"recortado en 2,5 % parecía {aparente:.0f} veces", fontsize=9.5, color=PALETA["acento"])
a1.set_xlim(0, por_localidad.max() * 1.15); a1.tick_params(axis="y", labelsize=9.5)
a1.set_xlabel("tasa de cancelación mensual registrada (%)", fontsize=9.5)
a1.set_title("1. El eje engañaba: las diferencias son pequeñas", loc="left", fontsize=12, pad=38)
x = np.arange(len(activos))
a2 = fig.add_subplot(gs[0, 1])
a2.plot(x, activos.values, color=PALETA["principal"], linewidth=2.5); a2.set_ylim(0, activos.max() * 1.2)
a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: miles(v))); a2.set_ylabel("clientes activos", fontsize=9.5)
a2.text(x[-1], activos.iloc[-1] * 1.06, f"{miles(activos.iloc[0])} → {miles(activos.iloc[-1])} "
        f"({decimal(100 * (activos.iloc[-1] / activos.iloc[0] - 1), 1)} %)", ha="right", fontsize=9.5)
a2.set_title("2. La cancelación es estable", loc="left", fontsize=12)
a3 = fig.add_subplot(gs[1, 1])
a3.axhspan(cancelacion.min(), cancelacion.max(), color=PALETA["fondo"], zorder=0)
a3.plot(x, cancelacion.values, color=PALETA["principal"], linewidth=2.5); a3.set_ylim(0, cancelacion.max() * 1.3)
a3.set_ylabel("cancelación (%)", fontsize=9.5)
a3.text(0, cancelacion.max() * 1.08, f"entre {decimal(cancelacion.min(), 1)} % y {decimal(cancelacion.max(), 1)} % todos los meses",
        fontsize=9.5, color=PALETA["secundario"])
for a in (a2, a3):
    a.set_xticks([0, 3, 6, 9, 11], ["nov. 25", "feb. 26", "may. 26", "ago. 26", "oct. 26"], fontsize=9.5)
a4 = fig.add_subplot(gs[:, 2])
top = fuera_loc.head(10).sort_values()
a4.barh([l.title() for l in top.index], top.values,
        color=[PALETA["acento"] if l == "KENNEDY" else PALETA["neutro"] for l in top.index], height=0.65)
for l, v in top.items():
    a4.text(v + 15, l.title(), miles(v), va="center", fontsize=9.5)
limpiar_ejes(a4); a4.set_xlim(0, top.max() * 1.25); a4.tick_params(axis="y", labelsize=9.5)
a4.set_xlabel("filas que el informe dejó fuera", fontsize=9.5)
a4.text(0, len(top) + 0.2, f"{miles(fuera.sum())} filas ({decimal(100 * fuera.mean(), 1)} %) con el nombre mal escrito;\n"
        f"cada localidad aparece escrita de {formas.max()} formas", fontsize=9.5, color=PALETA["secundario"])
a4.set_title("3. Los datos aún no están limpios", loc="left", fontsize=12, pad=38)
titulo(fig, "Lo que dejó la S1: las diferencias entre localidades son pequeñas, la cancelación es estable y los datos "
       "aún no están limpios", "CONECTA, datos sin revisar (nov. 2025 – oct. 2026); Kennedy, la que encabeza la tasa, "
       "es de las que más filas pierde", y=1.07)
fuente(fig, FUENTE_CONECTA)
guardar(fig, "s1_conecta.png")

## Errores frecuentes

1. **Una barra para mostrar una distribución.** La mediana por zona no dice cómo se reparten los distritos.
2. **Un histograma con un solo ancho de intervalo.** El ancho cambia la forma; hay que probar varios.
3. **Dispersión con miles de puntos sólidos.** Los puntos se tapan; transparencia o histograma en dos dimensiones.
4. **Múltiplos pequeños con una escala distinta en cada panel.** Grupos muy distintos parecen iguales.
5. **Tratar la edad como tiempo.** El censo es una foto; una línea por edad no es una tendencia.
6. **Coordenadas polares para una serie que no se repite.** El círculo une el último dato con el primero.
7. **Calcular un intervalo de confianza sin una muestra aleatoria.** Con todos los datos no hay error de muestreo;
   lo que importa es el tamaño de cada grupo: con pocos casos, la cifra es inestable.

## Síntesis

| Qué hicimos | Por qué importa |
|---|---|
| Definimos la razón precio/ingreso y la distinguimos de un KPI | Responde la pregunta del comité; un KPI necesita además meta y seguimiento |
| Recorrimos el directorio con sus galerías: cantidades, distribuciones, proporciones, relaciones, tiempo, geografía, incertidumbre | Cada tipo de dato tiene un gráfico de referencia y alternativas |
| Vimos los gráficos para presentar y los que conviene evitar | La audiencia de negocio necesita gráficos simples y honestos |
| Comparamos zonas con múltiplos pequeños | Con la misma escala, las diferencias entre grupos se ven como son |
| Manejamos la superposición de puntos | Con miles de casos, la dispersión engaña |
| Calculamos intervalos con muestras aleatorias y vimos qué es «95 % de confianza» | Una cifra estimada viene con su margen; con todos los datos, lo que pesa es el tamaño del grupo |
| Sacamos una primera lista de 1 032 «gangas», concentrada en el Valle Central | Tiene valores sospechosos: la S3 los revisa |

**Próxima sesión:** ¿podemos confiar en lo que vemos? Calidad de los datos.

## Transición al taller práctico

Cada equipo define el indicador de su proyecto y hace el mismo recorrido con CONECTA S.A. (proyecto A: Retención)
en `sesion_02_practica_proyecto_A.ipynb`: un gráfico por tipo de dato, con la serie mensual en múltiplos pequeños.
Parte de lo que dejó la S1 (Parte 8).

## Referencias

Diez, D. M., Çetinkaya-Rundel, M., y Barr, C. D. (2019). *OpenIntro statistics* (4.ª ed.). OpenIntro. https://www.openintro.org/book/os/

Knaflic, C. N. (2015). *Storytelling with data: A data visualization guide for business professionals*. Wiley.

Microsoft. (2026). *Visuales de indicador clave de rendimiento (KPI) - Power BI* [Microsoft Learn]. https://learn.microsoft.com/es-es/power-bi/visuals/power-bi-visualization-kpi

Pace, R. K., y Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters, 33*(3), 291–297. https://doi.org/10.1016/S0167-7152(96)00140-X

U.S. Census Bureau. (2019). *2018 cartographic boundary file, current county and equivalent for United States, 1:20,000,000* [Shapefile].

Ferdio. (s. f.). *Data Viz Project*. Recuperado el 13 de noviembre de 2026, de https://datavizproject.com/

Wilke, C. O. (2019). *Fundamentals of data visualization: A primer on making informative and compelling figures*. O'Reilly Media.